# 01 - Data Preparation

Compiles PII benchmark datasets for the ablation study.

<strong>HuggingFace Datasets:</strong>
- AI4Privacy, NVIDIA Nemotron-PII, Gretel PII masking

<strong>Experimental Design: 2x2 Factorial</strong>
- <strong>Zero-shot</strong>: No documentation, no tool access
- <strong>+Docs</strong>: Documentation only
- <strong>+Tool</strong>: Tool access only
- <strong>+Skills</strong>: Documentation + tool access

This design isolates the effects of documentation vs. tool access and measures their interaction.

<strong>Creates 2 HuggingFace repos:</strong>
- `pii-skills-ablation` - Benchmark (sample indices + ground truth) and config (prompts, config.yaml, mapping) under `config/`
- `pii-skills-ablation-results` - (empty, populated by Notebook 2)

<strong>Note:</strong> This notebook is provided for transparency and reproducibility. The compiled dataset is already available at `EdyVision/pii-skills-ablation`.

In [ ]:
%pip install -q datasets huggingface-hub pandas tqdm python-dotenv

In [3]:
# All imports for this notebook (run once after pip install)
import ast
import json
import os
import random
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, Features, Value
from dotenv import load_dotenv
from huggingface_hub import HfApi, login, create_repo, upload_file
from pii_codex.models.common import PIIType
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Repo root (notebook may be run from notebooks/ or repo root)
_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Load .env from repo root (for HF_TOKEN etc.)
_env_path = _repo_root / ".env"
if _env_path.exists():
    load_dotenv(_env_path)
    print("Loaded .env from", _env_path)
else:
    load_dotenv()
    print("No .env at repo root, using system env vars")

In [5]:
# Configuration from repo root config.yaml (single source of truth)
import yaml

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_config_path = _repo_root / "config.yaml"
if not _config_path.exists():
    raise FileNotFoundError(
        f"config.yaml not found at {_config_path}. Run from repo root or notebooks/."
    )
with open(_config_path) as f:
    CONFIG = yaml.safe_load(f)
# Ensure experiment.sampling exists for notebook (config.yaml has full structure)
if "experiment" not in CONFIG:
    CONFIG["experiment"] = {}
if "sampling" not in CONFIG["experiment"]:
    CONFIG["experiment"]["sampling"] = {
        "target_n": 1000,
        "pilot_n": 200,
        "max_n": 2000,
        "stratify_by": ["pii_types", "text_length_bin"],
    }

print(f"Experiment: {CONFIG['experiment']['name']}")
print(f"Target samples per dataset: {CONFIG['experiment']['sampling']['target_n']}")

# HuggingFace setup - login and create repos if needed
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("\nLogged in with HF_TOKEN")
else:
    print("\nNo HF_TOKEN found - run `huggingface-cli login` or set HF_TOKEN")

api = HfApi()
# Unique repos (benchmark and config share one repo: pii-skills-ablation)
all_repos = list(
    dict.fromkeys(
        [
            CONFIG["compiled_dataset"]["repo_id"],
            CONFIG["config_repo"]["repo_id"],
            CONFIG["results_repo"]["repo_id"],
        ]
    )
)
for repo_id in all_repos:
    try:
        api.repo_info(repo_id=repo_id, repo_type="dataset")
        print(f"Repo exists: {repo_id}")
    except Exception:
        print(f"Creating repo: {repo_id} (private)")
        create_repo(repo_id, repo_type="dataset", private=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Experiment: pii-skills-ablation
Target samples per dataset: 1000

Logged in with HF_TOKEN
Creating repo: EdyVision/pii-skills-ablation (private)
Creating repo: EdyVision/pii-skills-ablation-results (private)


## 1. AI4Privacy Dataset

In [6]:
print("Loading AI4Privacy dataset...")
ai4privacy = load_dataset("ai4privacy/pii-masking-300k", split="train")
# Restrict to English so downstream counts and stratification use English-only pool
ai4privacy = ai4privacy.filter(
    lambda x: (x.get("language") or "").strip().lower() == "english"
)
print(f"After language filter (English only): {len(ai4privacy):,} samples")

print(f"Total samples: {len(ai4privacy):,}")
print(f"Columns: {ai4privacy.column_names}")

# Inspect sample STRUCTURE only (no PII values)
sample = ai4privacy[0]
print("\nSample structure (first sentence):")
print(f"  source_text: <{len(sample['source_text'])} chars>")
print(f"  target_text: <{len(sample['target_text'])} chars>")
print(f"  privacy_mask: {len(sample['privacy_mask'])} PII items")
if sample["privacy_mask"]:
    # Show types only, not values
    types = [item.get("label", "UNK") for item in sample["privacy_mask"]]
    print(f"    Types found: {set(types)}")
print(f"  language: {sample.get('language', 'N/A')}")

Loading AI4Privacy dataset...
After language filter (English only): 29,908 samples
Total samples: 29,908
Columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']

Sample structure (first sentence):
  source_text: <454 chars>
  target_text: <430 chars>
  privacy_mask: 9 PII items
    Types found: {'USERNAME', 'TIME'}
  language: English


In [7]:
# AI4Privacy PII Distribution Diagnostic
pii_counts = []
for i in range(len(ai4privacy)):
    row = ai4privacy[i]
    pii_count = len(row.get("privacy_mask", []))
    pii_counts.append(pii_count)

print(f"AI4Privacy PII distribution ({len(ai4privacy):,} docs):")
print(
    f"  Docs with 0 PII: {sum(1 for c in pii_counts if c == 0):,} ({100*sum(1 for c in pii_counts if c == 0)/len(pii_counts):.1f}%)"
)
print(f"  Docs with 1+ PII: {sum(1 for c in pii_counts if c > 0):,}")
print(f"  Mean PII per doc: {np.mean(pii_counts):.2f}")
print(f"  Max PII per doc: {max(pii_counts)}")

# Dataset composition (imported so far)
df_ai4 = pd.DataFrame(
    [
        {
            "source": "ai4privacy",
            "path": "ai4privacy/pii-masking-300k",
            "n_rows": len(ai4privacy),
            "text_column": "source_text",
            "label_column": "privacy_mask",
            "docs_with_pii_pct": round(
                100 * sum(1 for c in pii_counts if c > 0) / len(pii_counts), 1
            ),
            "mean_pii_per_doc": round(np.mean(pii_counts), 2),
        }
    ]
)
display(df_ai4)

AI4Privacy PII distribution (29,908 docs):
  Docs with 0 PII: 3,482 (11.6%)
  Docs with 1+ PII: 26,426
  Mean PII per doc: 6.46
  Max PII per doc: 78


,source,path,n_rows,text_column,label_column,docs_with_pii_pct,mean_pii_per_doc
0,ai4privacy,ai4privacy/pii-masking-300k,29908,source_text,privacy_mask,88.4,6.46


## 2. NVIDIA Nemotron-PII Dataset

HuggingFace dataset with span-level PII labels (text + spans with start, end, label).

In [8]:
def _parse_list_or_json(val):
    """Parse column that may be list, Arrow sequence, JSON string, or Python repr string."""
    if val is None:
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, TypeError):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return []
    if hasattr(val, "__iter__") and not isinstance(val, (str, bytes)):
        return list(val)
    return []


print("Loading NVIDIA Nemotron-PII dataset...")
nvidia_ds = load_dataset("nvidia/Nemotron-PII", split="train")
# Restrict to US locale (Nemotron-PII uses 'us' | 'intl'; 'us' = US English)
nvidia_ds = nvidia_ds.filter(lambda x: (x.get("locale") or "").strip().lower() == "us")
print(f"After locale filter (us only): {len(nvidia_ds):,} samples")

print(f"Total samples: {len(nvidia_ds):,}")
print(f"Columns: {nvidia_ds.column_names}")

sample = nvidia_ds[0]
print("\nSample structure (row 0):")
print(f"  text: <{len(sample.get('text', ''))} chars>")
spans = _parse_list_or_json(sample.get("spans"))
print(f"  spans: {len(spans)} PII items")
if spans:
    types = []
    for s in spans:
        if isinstance(s, dict):
            types.append(
                s.get("label") or s.get("entity_type") or s.get("category") or "UNK"
            )
    if types:
        print(f"    Types in this sample: {set(types)}")
print(f"  (Full dataset has many more PII types across {len(nvidia_ds):,} docs.)")

Loading NVIDIA Nemotron-PII dataset...
After locale filter (us only): 50,000 samples
Total samples: 50,000
Columns: ['uid', 'domain', 'document_type', 'document_description', 'document_format', 'locale', 'text', 'spans', 'text_tagged']

Sample structure (row 0):
  text: <149 chars>
  spans: 4 PII items
    Types in this sample: {'employment_status', 'street_address', 'first_name', 'date_of_birth'}
  (Full dataset has many more PII types across 50,000 docs.)


In [9]:
# NVIDIA Nemotron-PII PII distribution diagnostic
pii_counts = []
for i in range(len(nvidia_ds)):
    row = nvidia_ds[i]
    spans = _parse_list_or_json(row.get("spans"))
    count = 0
    for s in spans:
        if isinstance(s, dict) and (
            s.get("label") or s.get("entity_type") or s.get("category")
        ):
            count += 1
    pii_counts.append(count)

print(f"NVIDIA Nemotron-PII PII distribution ({len(nvidia_ds):,} docs):")
print(
    f"  Docs with 0 PII: {sum(1 for c in pii_counts if c == 0):,} ({100*sum(1 for c in pii_counts if c == 0)/len(pii_counts):.1f}%)"
)
print(f"  Docs with 1+ PII: {sum(1 for c in pii_counts if c > 0):,}")
print(f"  Mean PII per doc: {np.mean(pii_counts):.2f}")
print(f"  Max PII per doc: {max(pii_counts)}")

df_nvidia = pd.DataFrame(
    [
        {
            "source": "nvidia_nemotron_pii",
            "path": "nvidia/Nemotron-PII",
            "n_rows": len(nvidia_ds),
            "text_column": "text",
            "label_column": "spans",
            "docs_with_pii_pct": round(
                100 * sum(1 for c in pii_counts if c > 0) / len(pii_counts), 1
            ),
            "mean_pii_per_doc": round(np.mean(pii_counts), 2),
        }
    ]
)
display(df_nvidia)

NVIDIA Nemotron-PII PII distribution (50,000 docs):
  Docs with 0 PII: 2 (0.0%)
  Docs with 1+ PII: 49,998
  Mean PII per doc: 8.28
  Max PII per doc: 149


,source,path,n_rows,text_column,label_column,docs_with_pii_pct,mean_pii_per_doc
0,nvidia_nemotron_pii,nvidia/Nemotron-PII,50000,text,spans,100.0,8.28


## 3. Gretel PII Masking Dataset

HuggingFace dataset with entity-level PII (text + entities list with entity string and types).

In [10]:
print("Loading Gretel PII masking dataset...")
gretel_ds = load_dataset("gretelai/gretel-pii-masking-en-v1", split="train")
# No locale/language filter: dataset is English-only (gretel-pii-masking-en-v1)

print(f"Total samples: {len(gretel_ds):,}")
print(f"Columns: {gretel_ds.column_names}")

sample = gretel_ds[0]
print("\nSample structure (row 0):")
print(f"  text: <{len(sample.get('text', ''))} chars>")
entities = _parse_list_or_json(sample.get("entities"))
print(f"  entities: {len(entities)} PII items")
if entities:
    types = []
    for e in entities:
        if isinstance(e, dict):
            t = e.get("types") or e.get("type") or []
            types.extend(t if isinstance(t, list) else [t])
    if types:
        print(f"    Types in this sample: {set(types)}")
print(f"  (Full dataset has many more PII types across {len(gretel_ds):,} docs.)")

Loading Gretel PII masking dataset...
Total samples: 50,000
Columns: ['uid', 'domain', 'document_type', 'document_description', 'entities', 'text']

Sample structure (row 0):
  text: <237 chars>
  entities: 4 PII items
    Types in this sample: {'name', 'date_of_birth', 'country', 'unique_identifier'}
  (Full dataset has many more PII types across 50,000 docs.)


In [11]:
# Gretel PII masking PII distribution diagnostic
pii_counts = []
for i in range(len(gretel_ds)):
    row = gretel_ds[i]
    entities = _parse_list_or_json(row.get("entities"))
    count = 0
    for e in entities:
        if isinstance(e, dict):
            t = e.get("types") or e.get("type") or []
            count += len(t) if isinstance(t, list) else (1 if t else 0)
    pii_counts.append(count)

print(f"Gretel PII masking PII distribution ({len(gretel_ds):,} docs):")
print(
    f"  Docs with 0 PII: {sum(1 for c in pii_counts if c == 0):,} ({100*sum(1 for c in pii_counts if c == 0)/len(pii_counts):.1f}%)"
)
print(f"  Docs with 1+ PII: {sum(1 for c in pii_counts if c > 0):,}")
print(f"  Mean PII per doc: {np.mean(pii_counts):.2f}")
print(f"  Max PII per doc: {max(pii_counts)}")

df_gretel = pd.DataFrame(
    [
        {
            "source": "gretel_pii_masking",
            "path": "gretelai/gretel-pii-masking-en-v1",
            "n_rows": len(gretel_ds),
            "text_column": "text",
            "label_column": "entities",
            "docs_with_pii_pct": round(
                100 * sum(1 for c in pii_counts if c > 0) / len(pii_counts), 1
            ),
            "mean_pii_per_doc": round(np.mean(pii_counts), 2),
        }
    ]
)
display(df_gretel)

Gretel PII masking PII distribution (50,000 docs):
  Docs with 0 PII: 0 (0.0%)
  Docs with 1+ PII: 50,000
  Mean PII per doc: 4.24
  Max PII per doc: 19


,source,path,n_rows,text_column,label_column,docs_with_pii_pct,mean_pii_per_doc
0,gretel_pii_masking,gretelai/gretel-pii-masking-en-v1,50000,text,entities,100.0,4.24


## 4. Stratified Sample Selection

Stratified sampling by PII type presence and text length bins to ensure representativeness.

In [12]:
# nvidia_ds and gretel_ds are loaded in Sections 2 and 3
sampling_config = CONFIG["experiment"]["sampling"]
target_n = sampling_config["target_n"]
seed = CONFIG["experiment"]["seed"]
MIN_LABELED_RECORDS = 2000
# Three sources only; oversample so after excluding empty/unmappable we still reach MIN_LABELED_RECORDS
pool_n = max(target_n, (MIN_LABELED_RECORDS + 600) // 3)

print(f"Sampling Strategy:")
print(
    f"  Target: {target_n} samples per dataset (pool: {pool_n} per source for >= {MIN_LABELED_RECORDS} after exclusions)"
)
print(f"  Seed: {seed}")
print(f"  Stratify by: {sampling_config['stratify_by']}")


def get_text_length_bin(text: str) -> str:
    """Bin text by character length."""
    length = len(text)
    if length < 500:
        return "short"
    elif length < 2000:
        return "medium"
    return "long"


def get_pii_signature(ground_truth: list) -> str:
    """Create a signature of PII types present."""
    if not ground_truth:
        return "none"
    types = sorted(set(gt.get("type", "UNK") for gt in ground_truth))
    return "|".join(types[:3])


def stratified_sample(samples: list, n: int, seed: int) -> list:
    """Stratified sampling by text length and PII types."""
    random.seed(seed)

    for s in samples:
        s["_length_bin"] = get_text_length_bin(s["text"])
        s["_pii_sig"] = get_pii_signature(s.get("ground_truth", []))
        s["_stratum"] = f"{s['_length_bin']}_{s['_pii_sig']}"

    strata = {}
    for s in samples:
        stratum = s["_stratum"]
        if stratum not in strata:
            strata[stratum] = []
        strata[stratum].append(s)

    print(f"  Found {len(strata)} strata")

    total = len(samples)
    selected = []
    for stratum, stratum_samples in strata.items():
        share = max(1, int(n * len(stratum_samples) / total))
        share = min(share, len(stratum_samples))
        random.shuffle(stratum_samples)
        selected.extend(stratum_samples[:share])

    remaining = [s for s in samples if s not in selected]
    random.shuffle(remaining)
    while len(selected) < n and remaining:
        selected.append(remaining.pop())

    random.shuffle(selected)
    selected = selected[:n]

    for s in selected:
        del s["_length_bin"], s["_pii_sig"], s["_stratum"]

    return selected


def normalize_ai4privacy(dataset, n: int) -> list:
    """Convert AI4Privacy samples to dehydrated format (indices only)."""
    samples = []
    indices = random.sample(range(len(dataset)), min(n * 5, len(dataset)))
    for idx in indices:
        row = dataset[idx]
        # Parse privacy_mask for ground truth (needed for stratification, not stored)
        gt = []
        if row.get("privacy_mask"):
            for item in row["privacy_mask"]:
                if isinstance(item, dict):
                    gt.append({"type": item.get("label", "UNK")})
        # Store index and text temporarily for stratification
        samples.append(
            {
                "id": f"ai4_{idx}",
                "original_index": idx,
                "text": row["source_text"],  # Temporary, removed before upload
                "ground_truth": gt,  # Temporary, removed before upload
            }
        )
    return samples


def _parse_list_or_json(val):
    """Parse column that may be list, Arrow sequence, JSON string, or Python repr string."""
    if val is None:
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, TypeError):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return []
    if hasattr(val, "__iter__") and not isinstance(val, (str, bytes)):
        return list(val)
    return []


def normalize_nvidia(dataset, n: int) -> list:
    """Convert NVIDIA Nemotron-PII to dehydrated format (text + ground_truth types)."""
    samples = []
    indices = random.sample(range(len(dataset)), min(n * 5, len(dataset)))
    for idx in indices:
        row = dataset[idx]
        gt = []
        spans = _parse_list_or_json(row.get("spans"))
        for s in spans:
            if not isinstance(s, dict) and hasattr(s, "__getitem__"):
                try:
                    s = dict(s)
                except (TypeError, ValueError):
                    s = None
            if isinstance(s, dict):
                label = (
                    s.get("label") or s.get("entity_type") or s.get("category") or "UNK"
                )
                gt.append({"type": label})
            elif isinstance(s, str):
                try:
                    s = (
                        json.loads(s)
                        if s.strip().startswith("{")
                        else ast.literal_eval(s)
                    )
                    if isinstance(s, dict):
                        gt.append({"type": s.get("label", "UNK")})
                except (json.JSONDecodeError, ValueError, SyntaxError):
                    pass
        samples.append(
            {
                "id": f"nvidia_{idx}",
                "original_index": idx,
                "text": row.get("text", ""),
                "ground_truth": gt,
            }
        )
    return samples


def normalize_gretel(dataset, n: int) -> list:
    """Convert Gretel PII masking to dehydrated format (text + ground_truth types)."""
    samples = []
    indices = random.sample(range(len(dataset)), min(n * 5, len(dataset)))
    for idx in indices:
        row = dataset[idx]
        gt = []
        entities = _parse_list_or_json(row.get("entities"))
        for e in entities:
            if not isinstance(e, dict) and hasattr(e, "__getitem__"):
                try:
                    e = dict(e)
                except (TypeError, ValueError):
                    e = None
            if isinstance(e, dict):
                types = e.get("types") or e.get("type") or []
                if isinstance(types, str):
                    types = [types]
                for t in types:
                    gt.append({"type": t})
            elif isinstance(e, str):
                try:
                    e = (
                        json.loads(e)
                        if e.strip().startswith("{")
                        else ast.literal_eval(e)
                    )
                    if isinstance(e, dict):
                        for t in e.get("types") or []:
                            gt.append({"type": t})
                except (json.JSONDecodeError, ValueError, SyntaxError):
                    pass
        samples.append(
            {
                "id": f"gretel_{idx}",
                "original_index": idx,
                "text": row.get("text", ""),
                "ground_truth": gt,
            }
        )
    return samples


# Sample from each dataset
random.seed(seed)
samples_by_dataset = {}

print("\nAI4Privacy...")
raw = normalize_ai4privacy(ai4privacy, pool_n)
samples_by_dataset["ai4privacy"] = stratified_sample(raw, pool_n, seed)
print(f"  Selected: {len(samples_by_dataset['ai4privacy'])} samples")

print("\nNVIDIA Nemotron-PII...")
raw = normalize_nvidia(nvidia_ds, pool_n)
samples_by_dataset["nvidia_nemotron_pii"] = stratified_sample(raw, pool_n, seed)
print(f"  Selected: {len(samples_by_dataset['nvidia_nemotron_pii'])} samples")

print("\nGretel PII masking...")
raw = normalize_gretel(gretel_ds, pool_n)
samples_by_dataset["gretel_pii_masking"] = stratified_sample(raw, pool_n, seed)
print(f"  Selected: {len(samples_by_dataset['gretel_pii_masking'])} samples")

# Dataset composition (all imported)
rows = [
    {
        "source": "ai4privacy",
        "path": "ai4privacy/pii-masking-300k",
        "n_rows": len(ai4privacy),
        "text_column": "source_text",
        "label_column": "privacy_mask",
    },
    {
        "source": "nvidia_nemotron_pii",
        "path": "nvidia/Nemotron-PII",
        "n_rows": len(nvidia_ds),
        "text_column": "text",
        "label_column": "spans",
    },
    {
        "source": "gretel_pii_masking",
        "path": "gretelai/gretel-pii-masking-en-v1",
        "n_rows": len(gretel_ds),
        "text_column": "text",
        "label_column": "entities",
    },
]
df_imported = pd.DataFrame(rows)
display(df_imported)

# Final dataset composition (stratified sample)
final_rows = [
    {
        "source": name,
        "n_samples": len(samples),
        "pct_of_total": round(
            100 * len(samples) / sum(len(s) for s in samples_by_dataset.values()), 1
        ),
    }
    for name, samples in samples_by_dataset.items()
]
df_final = pd.DataFrame(final_rows)
display(df_final)

Sampling Strategy:
  Target: 1000 samples per dataset (pool: 1000 per source for >= 2000 after exclusions)
  Seed: 42
  Stratify by: ['pii_types', 'text_length_bin']

AI4Privacy...
  Found 1068 strata
  Selected: 1000 samples

NVIDIA Nemotron-PII...
  Found 2188 strata
  Selected: 1000 samples

Gretel PII masking...
  Found 952 strata
  Selected: 1000 samples


,source,path,n_rows,text_column,label_column
0,ai4privacy,ai4privacy/pii-masking-300k,29908,source_text,privacy_mask
1,nvidia_nemotron_pii,nvidia/Nemotron-PII,50000,text,spans
2,gretel_pii_masking,gretelai/gretel-pii-masking-en-v1,50000,text,entities


,source,n_samples,pct_of_total
0,ai4privacy,1000,33.3
1,nvidia_nemotron_pii,1000,33.3
2,gretel_pii_masking,1000,33.3


## 5. Verify Sample Quality

In [13]:
def summarize_samples(samples, name):
    """Print summary statistics for samples."""
    if not samples:
        print(f"{name}: No samples available")
        return

    text_lengths = [len(s["text"]) for s in samples]
    pii_counts = [len(s.get("ground_truth", [])) for s in samples]

    print(f"\n{name}:")
    print(f"  Samples: {len(samples)}")
    print(
        f"  Text length: min={min(text_lengths)}, max={max(text_lengths)}, mean={sum(text_lengths)/len(text_lengths):.0f}"
    )
    print(
        f"  PII per sample: min={min(pii_counts)}, max={max(pii_counts)}, mean={sum(pii_counts)/len(pii_counts):.1f}"
    )

    # PII type distribution
    all_types = []
    for s in samples:
        for gt in s.get("ground_truth", []):
            all_types.append(gt.get("type", "UNKNOWN"))

    type_counts = Counter(all_types)
    print(f"  PII types: {dict(type_counts.most_common(10))}")


# Summarize each dataset
for name, samples in samples_by_dataset.items():
    summarize_samples(samples, name)


ai4privacy:
  Samples: 1000
  Text length: min=262, max=512, mean=432
  PII per sample: min=0, max=33, mean=6.7
  PII types: {'TIME': 536, 'USERNAME': 446, 'SOCIALNUMBER': 384, 'EMAIL': 335, 'LASTNAME1': 329, 'IP': 327, 'TEL': 309, 'BOD': 307, 'DRIVERLICENSE': 305, 'PASSPORT': 295}

nvidia_nemotron_pii:
  Samples: 1000
  Text length: min=77, max=4747, mean=969
  PII per sample: min=2, max=46, mean=7.8
  PII types: {'date': 578, 'company_name': 573, 'first_name': 525, 'url': 451, 'last_name': 394, 'occupation': 349, 'email': 323, 'time': 289, 'phone_number': 224, 'street_address': 188}

gretel_pii_masking:
  Samples: 1000
  Text length: min=20, max=1302, mean=234
  PII per sample: min=1, max=14, mean=4.3
  PII types: {'medical_record_number': 461, 'date_of_birth': 421, 'ssn': 320, 'date': 223, 'email': 214, 'first_name': 193, 'employee_id': 193, 'street_address': 189, 'customer_id': 178, 'ipv4': 176}


## 6. Compile Dehydrated Dataset

Create a dehydrated dataset (indices + ground truth labels only, no raw text) due to licensing restrictions. Text is fetched via hydration in notebook 2.

**PII Codex labels:** Source dataset labels are mapped to PII Codex canonical types (`pii_codex_ground_truth`) so the benchmark can be used standalone with PII Codex and experiments do not need to map at evaluation time.

In [14]:
# Load PII label -> PII Codex mapping from config (single source of truth; uploaded to config repo in Section 9)
mapping_path = Path("../config/pii_label_to_piicodex.json")
with open(mapping_path) as f:
    _raw = json.load(f)
SOURCE_LABEL_TO_PII_CODEX = {k: v for k, v in _raw.items()}  # v is str or None


def map_source_label_to_pii_codex(label: str):
    """Map a source dataset PII label to PII Codex PIIType.name, or None if not mappable."""
    if not label:
        return None
    normalized = str(label).upper().replace("-", "_").replace(" ", "_")
    out = SOURCE_LABEL_TO_PII_CODEX.get(normalized)
    if out is not None:
        return out
    # If label matches a PIIType.name exactly, use it
    if normalized in [e.name for e in PIIType]:
        return normalized
    return None

In [15]:
# Compile DEHYDRATED samples (no raw text due to licensing restrictions)
# Ground truth contains only PII type labels, not actual PII values
# pii_codex_ground_truth = same labels mapped to PII Codex PIIType.name for standalone use

# Diagnostic: ground-truth labels that do not map to PII Codex (add to SOURCE_LABEL_TO_PII_CODEX)
label_to_sources = defaultdict(set)
label_counts = Counter()
for source_name, samples in samples_by_dataset.items():
    for sample in samples:
        for g in sample.get("ground_truth", []):
            raw = g.get("type", "")
            norm = raw.upper().replace("-", "_").replace(" ", "_") if raw else ""
            if not norm:
                continue
            mapped = map_source_label_to_pii_codex(raw)
            if mapped is None:
                label_counts[norm] += 1
                label_to_sources[norm].add(source_name)
if label_counts:
    rows = [
        {
            "source_label": label,
            "count": label_counts[label],
            "sources": ", ".join(sorted(label_to_sources[label])),
        }
        for label in sorted(label_counts.keys())
    ]
    df_unknowns = pd.DataFrame(rows)
    print(
        "Ground-truth labels with no PII Codex mapping (add to SOURCE_LABEL_TO_PII_CODEX in the cell above):"
    )
    display(df_unknowns)
    print(f"Total unmapped instances: {sum(label_counts.values())}")
else:
    print("All ground-truth labels map to PII Codex. No unknowns.")

compiled_samples = []
sample_stats = []  # Per-sample stats (text length, PII counts) before we drop text
excluded_unmappable = (
    0  # Rows dropped because they contain at least one unmappable PII type
)
excluded_empty = 0  # Rows with no ground_truth (must not ship)

for source_name, samples in samples_by_dataset.items():
    for sample in samples:
        gt = sample.get("ground_truth", [])
        if not gt:
            excluded_empty += 1
            continue
        mapped = [map_source_label_to_pii_codex(g.get("type")) for g in gt]
        pii_codex_types = [t for t in mapped if t is not None]
        if len(pii_codex_types) < len(gt):
            excluded_unmappable += 1
            continue
        text = sample.get("text", "")
        compiled_samples.append(
            {
                "id": sample["id"],
                "source": source_name,
                "original_index": sample["original_index"],
                # NO TEXT - must be hydrated from original sources in notebook 2
                "ground_truth": gt,  # Original source type labels
                "pii_codex_ground_truth": pii_codex_types,  # PII Codex PIIType.name for evaluation/standalone use
            }
        )
        sample_stats.append(
            {
                "source": source_name,
                "id": sample["id"],
                "n_pii": len(gt),
                "n_pii_codex": len(pii_codex_types),
                "text_chars": len(text),
                "text_words": len(text.split()) if text else 0,
            }
        )

if excluded_empty:
    print(
        f"Excluded {excluded_empty} samples with no ground_truth (fully labeled only)."
    )
if excluded_unmappable:
    print(
        f"Excluded {excluded_unmappable} samples with unmappable PII instances (only fully mappable rows included)."
    )
# Safety: drop any remaining empty ground_truth (should be zero after loop)
keep_idx = [
    i for i, s in enumerate(compiled_samples) if len(s.get("ground_truth", [])) > 0
]
compiled_samples = [compiled_samples[i] for i in keep_idx]
sample_stats = [sample_stats[i] for i in keep_idx]
print(
    f"Compiled {len(compiled_samples)} total samples (all with non-empty ground_truth)"
)
print(
    f"  - ai4privacy: {len([s for s in compiled_samples if s['source'] == 'ai4privacy'])}"
)
print(
    f"  - nvidia_nemotron_pii: {len([s for s in compiled_samples if s['source'] == 'nvidia_nemotron_pii'])}"
)
print(
    f"  - gretel_pii_masking: {len([s for s in compiled_samples if s['source'] == 'gretel_pii_masking'])}"
)

# Ensure at least 2000 labeled records: cap at 2000 or top up from extra draws
MIN_LABELED_RECORDS = 2000
compiled_ids = {(s["source"], s["original_index"]) for s in compiled_samples}
if len(compiled_samples) >= MIN_LABELED_RECORDS:
    compiled_samples = compiled_samples[:MIN_LABELED_RECORDS]
    sample_stats = sample_stats[:MIN_LABELED_RECORDS]
else:
    need = MIN_LABELED_RECORDS - len(compiled_samples)
    extra_draws = [
        ("ai4privacy", normalize_ai4privacy(ai4privacy, 1200), 1200),
        ("nvidia_nemotron_pii", normalize_nvidia(nvidia_ds, 1200), 1200),
        ("gretel_pii_masking", normalize_gretel(gretel_ds, 1200), 1200),
    ]
    for source_name, raw_list, n in extra_draws:
        if need <= 0:
            break
        extra_samples = stratified_sample(raw_list, n, seed + 1)
        for sample in extra_samples:
            if need <= 0:
                break
            key = (source_name, sample["original_index"])
            if key in compiled_ids:
                continue
            gt = sample.get("ground_truth", [])
            if not gt:
                continue
            mapped = [map_source_label_to_pii_codex(g.get("type")) for g in gt]
            pii_codex_types = [t for t in mapped if t is not None]
            if len(pii_codex_types) < len(gt):
                continue
            compiled_ids.add(key)
            text = sample.get("text", "")
            compiled_samples.append(
                {
                    "id": sample["id"],
                    "source": source_name,
                    "original_index": sample["original_index"],
                    "ground_truth": gt,
                    "pii_codex_ground_truth": pii_codex_types,
                }
            )
            sample_stats.append(
                {
                    "source": source_name,
                    "id": sample["id"],
                    "n_pii": len(gt),
                    "n_pii_codex": len(pii_codex_types),
                    "text_chars": len(text),
                    "text_words": len(text.split()) if text else 0,
                }
            )
            need -= 1
    if need > 0:
        print(
            f"Warning: only {len(compiled_samples)} fully mappable after top-up (target {MIN_LABELED_RECORDS})."
        )

# Convert to HuggingFace Dataset
# Define schema - DEHYDRATED (indices + ground_truth + pii_codex_ground_truth)
features = Features(
    {
        "id": Value("string"),
        "source": Value("string"),
        "original_index": Value("int64"),  # Index to fetch from original dataset
        "ground_truth": Value("string"),  # JSON list of {type: <source label>}
        "pii_codex_ground_truth": Value(
            "string"
        ),  # JSON list of PII Codex PIIType.name for evaluation
    }
)

# Serialize list columns to JSON strings
for sample in compiled_samples:
    sample["ground_truth"] = json.dumps(sample["ground_truth"])
    sample["pii_codex_ground_truth"] = json.dumps(sample["pii_codex_ground_truth"])

compiled_dataset = Dataset.from_list(compiled_samples, features=features)
# Uniqueness: (source, original_index) is unique (main loop + top-up skip duplicates)
unique_keys = set((r["source"], r["original_index"]) for r in compiled_dataset)
assert len(unique_keys) == len(
    compiled_dataset
), "Duplicate (source, original_index) in compiled dataset"
print(f"\nDataset created: {compiled_dataset}")
print(f"  Unique (source, original_index): {len(unique_keys)}")

# Per-source summary (rich benchmark stats)
df_sample_stats = pd.DataFrame(sample_stats)
total = len(compiled_samples)
agg = (
    df_sample_stats.groupby("source")
    .agg(
        n_samples=("id", "count"),
        total_pii=("n_pii", "sum"),
        mean_pii_per_sample=("n_pii", "mean"),
        median_pii_per_sample=("n_pii", "median"),
        total_pii_codex=("n_pii_codex", "sum"),
        mean_pii_codex_per_sample=("n_pii_codex", "mean"),
        mean_text_chars=("text_chars", "mean"),
        median_text_chars=("text_chars", "median"),
        mean_text_words=("text_words", "mean"),
        median_text_words=("text_words", "median"),
    )
    .reset_index()
)
agg["pct_of_total"] = (agg["n_samples"] / total * 100).round(1)
agg = agg.round(
    {
        "mean_pii_per_sample": 2,
        "median_pii_per_sample": 1,
        "mean_pii_codex_per_sample": 2,
        "mean_text_chars": 0,
        "median_text_chars": 0,
        "mean_text_words": 1,
        "median_text_words": 1,
    }
)
df_benchmark = agg
display(df_benchmark)

# Per-sample stats (first 15 + describe)
print("\nPer-sample stats (sample):")
display(df_sample_stats.head(15))
print("\nPer-sample stats (describe):")
display(df_sample_stats.describe())

Ground-truth labels with no PII Codex mapping (add to SOURCE_LABEL_TO_PII_CODEX in the cell above):


,source_label,count,sources
0,BIOMETRIC_IDENTIFIER,190,"gretel_pii_masking, nvidia_nemotron_pii"
1,BLOOD_TYPE,62,nvidia_nemotron_pii
2,COMPANY_NAME,622,"gretel_pii_masking, nvidia_nemotron_pii"
3,CUSTOMER_ID,325,"gretel_pii_masking, nvidia_nemotron_pii"
4,DEVICE_IDENTIFIER,134,"gretel_pii_masking, nvidia_nemotron_pii"
5,EDUCATION_LEVEL,127,nvidia_nemotron_pii
6,EMPLOYEE_ID,324,"gretel_pii_masking, nvidia_nemotron_pii"
7,EMPLOYMENT_STATUS,81,nvidia_nemotron_pii
8,HTTP_COOKIE,71,nvidia_nemotron_pii
9,IDCARD,271,ai4privacy


Total unmapped instances: 2880
Excluded 69 samples with no ground_truth (fully labeled only).
Excluded 1491 samples with unmappable PII instances (only fully mappable rows included).
Compiled 1440 total samples (all with non-empty ground_truth)
  - ai4privacy: 572
  - nvidia_nemotron_pii: 287
  - gretel_pii_masking: 581
  Found 1168 strata

Dataset created: Dataset({
    features: ['id', 'source', 'original_index', 'ground_truth', 'pii_codex_ground_truth'],
    num_rows: 2000
})
  Unique (source, original_index): 2000


,source,n_samples,total_pii,mean_pii_per_sample,median_pii_per_sample,total_pii_codex,mean_pii_codex_per_sample,mean_text_chars,median_text_chars,mean_text_words,median_text_words,pct_of_total
0,ai4privacy,1132,7112,6.28,5.0,7112,6.28,434.0,438.0,50.2,53.0,56.6
1,gretel_pii_masking,581,2416,4.16,4.0,2416,4.16,229.0,181.0,29.9,22.0,29.0
2,nvidia_nemotron_pii,287,1842,6.42,6.0,1842,6.42,808.0,607.0,106.5,78.0,14.4



Per-sample stats (sample):


,source,id,n_pii,n_pii_codex,text_chars,text_words
0,ai4privacy,ai4_7055,12,12,477,53
1,ai4privacy,ai4_5293,3,3,427,40
2,ai4privacy,ai4_6407,2,2,508,69
3,ai4privacy,ai4_4730,5,5,383,50
4,ai4privacy,ai4_25346,11,11,500,37
5,ai4privacy,ai4_3773,8,8,461,16
6,ai4privacy,ai4_1993,3,3,461,71
7,ai4privacy,ai4_6219,2,2,447,66
8,ai4privacy,ai4_4550,6,6,383,16
9,ai4privacy,ai4_5556,10,10,425,35



Per-sample stats (describe):


,n_pii,n_pii_codex,text_chars,text_words
count,2000.000000,2000.000000,2000.000000,2000.000000
mean,5.685000,5.685000,427.996500,52.372500
std,3.954187,3.954187,333.112946,45.450661
min,1.000000,1.000000,20.000000,1.000000
25%,3.000000,3.000000,280.000000,27.000000
50%,5.000000,5.000000,415.500000,48.000000
75%,7.000000,7.000000,480.000000,63.000000
max,46.000000,46.000000,4747.000000,686.000000


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Base palette
EXTENDED_COLORS = [
    # Slate family
    "#2D353B",  # very dark slate
    "#4A5259",  # dark slate
    "#7A8A94",  # medium slate
    "#A8B5BC",  # light slate
    # Teal family
    "#1E3A47",  # very dark teal
    "#3D5A6C",  # deep teal
    "#5A8799",  # medium teal
    "#8FB4C4",  # light teal
    # Sage family
    "#4A5A3A",  # dark olive sage
    "#8B9E6B",  # muted sage
    "#A8BC8A",  # medium sage
    "#C8DBAF",  # light sage
    # Taupe family
    "#5C4D3F",  # dark taupe
    "#9A8C7D",  # warm taupe
    "#B8A998",  # medium taupe
    "#D4C9BC",  # light taupe
    # Accent fills
    "#6B5B7A",  # dusty purple
    "#7A6B5B",  # brown
    "#5B6B6B",  # gray-teal
    "#8A7B6A",  # tan
    "#6A7B6A",  # gray-green
    "#7B6A6A",  # mauve
    "#5A6A7A",  # steel blue
    "#7A7A5A",  # olive
]

# Light tints (half size, not too close to white) so violin fill doesn't hide inner box/mean line
LIGHT_EXTENDED_COLORS = [
    "#B8C0C8",
    "#A8B4BC",
    "#A0BCC8",
    "#98B4C0",  # slate / teal
    "#B0C0A0",
    "#B8C8A8",
    "#C0B0A0",
    "#C8BCB0",  # sage / taupe
    "#B8B0C0",
    "#C0B8B0",
    "#B0B8B8",
    "#C0B8A8",  # accent
]

# Original dataset sizes (from loaded datasets)
original_sizes = {
    "ai4privacy": len(ai4privacy),
    "nvidia_nemotron_pii": len(nvidia_ds),
    "gretel_pii_masking": len(gretel_ds),
}
sampled = CONFIG["experiment"]["sampling"]["target_n"]

sources = list(original_sizes.keys())

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Sampling from Source Datasets", "PII Density per Sample"),
    column_widths=[0.4, 0.6],
    horizontal_spacing=0.12,
)

# Left: Original vs sampled (grouped bar)
fig.add_trace(
    go.Bar(
        name="Original",
        x=sources,
        y=[original_sizes[s] for s in sources],
        marker_color=EXTENDED_COLORS[4],
        text=[f"{original_sizes[s]:,}" for s in sources],
        textposition="outside",
        opacity=0.5,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        name=f"Sampled ({sampled})",
        x=sources,
        y=[sampled] * len(sources),
        marker_color=EXTENDED_COLORS[5],
        text=[str(sampled)] * len(sources),
        textposition="inside",
    ),
    row=1,
    col=1,
)

# Right: Violin for PII density (unchanged)
for i, source in enumerate(sources):
    source_data = df_sample_stats[df_sample_stats["source"] == source]
    fig.add_trace(
        go.Violin(
            x=[source] * len(source_data),
            y=source_data["n_pii"],
            name=source,
            fillcolor=LIGHT_EXTENDED_COLORS[(i + 4) % len(LIGHT_EXTENDED_COLORS)],
            line_color=EXTENDED_COLORS[5],
            opacity=0.7,
            meanline_visible=True,
            box_visible=True,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

fig.update_layout(
    title="Benchmark Dataset Overview",
    plot_bgcolor="#fafafa",
    height=450,
    width=1050,
    barmode="overlay",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.2),
)

fig.update_yaxes(title_text="Samples", gridcolor="#e0e0e0", row=1, col=1)
fig.update_yaxes(
    title_text="PII Entities per Sample", gridcolor="#e0e0e0", row=1, col=2
)
fig.update_xaxes(tickangle=30)

fig.show(renderer="svg")
# fig.write_html("results/benchmark_overview.html")

In [ ]:
from collections import defaultdict, Counter

# PII type coverage
type_by_source = defaultdict(Counter)
for s in compiled_samples:
    source = s["source"]
    types = json.loads(s["pii_codex_ground_truth"])
    for t in types:
        type_by_source[source][t] += 1

sources = sorted(type_by_source.keys())
all_types = sorted(set(t for c in type_by_source.values() for t in c.keys()))

fig_type = go.Figure()
for i, pii_type in enumerate(all_types):
    counts = [type_by_source[src][pii_type] for src in sources]
    fig_type.add_trace(
        go.Bar(
            name=pii_type,
            x=counts,
            y=sources,
            orientation="h",
            marker_color=EXTENDED_COLORS[i % len(EXTENDED_COLORS)],
        )
    )

fig_type.update_layout(
    barmode="stack",
    title="PII Type Coverage by Source",
    xaxis_title="Count",
    yaxis_title="",
    height=550,  # taller to fit legend
    width=950,
    plot_bgcolor="#fafafa",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.25,  # push further down
        xanchor="center",
        x=0.5,
        font=dict(size=8),
        itemwidth=30,  # compact items
        traceorder="normal",
    ),
    margin=dict(l=140, b=180),  # more bottom margin
    xaxis=dict(title=dict(standoff=10)),  # keep axis label close to axis
)

fig_type.show(renderer="svg")
# fig_type.write_html("results/pii_type_coverage_by_source.html")

## 7. Upload to HuggingFace

Push the compiled dataset to HuggingFace Hub for use in notebook 2.

In [18]:
repo_id = CONFIG["compiled_dataset"]["repo_id"]
print(f"Uploading to: {repo_id}")

compiled_dataset.push_to_hub(
    repo_id,
    split="test",
    private=False,
    commit_message=f"Upload compiled PII benchmark ({len(compiled_samples)} samples)",
)

print(f"\nUploaded: https://huggingface.co/datasets/{repo_id}")

Uploading to: EdyVision/pii-skills-ablation


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Uploaded: https://huggingface.co/datasets/EdyVision/pii-skills-ablation


## 8. Verify Benchmark Upload

Quick sanity check that we can load the dataset back.

In [19]:
# Verify we can load it back
verification_ds = load_dataset(repo_id, split="test")
print(f"Verification: Loaded {len(verification_ds)} samples from HuggingFace")
print(f"Columns: {verification_ds.column_names}")
print(f"\nSample:\n{verification_ds[0]}")

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Verification: Loaded 2000 samples from HuggingFace
Columns: ['id', 'source', 'original_index', 'ground_truth', 'pii_codex_ground_truth']

Sample:
{'id': 'ai4_7055', 'source': 'ai4privacy', 'original_index': 7055, 'ground_truth': '[{"type": "BUILDING"}, {"type": "STREET"}, {"type": "CITY"}, {"type": "STATE"}, {"type": "POSTCODE"}, {"type": "IP"}, {"type": "BUILDING"}, {"type": "STREET"}, {"type": "CITY"}, {"type": "STATE"}, {"type": "POSTCODE"}, {"type": "IP"}]', 'pii_codex_ground_truth': '["ADDRESS", "ADDRESS", "LOCATION", "LOCATION", "ZIPCODE", "IP_ADDRESS", "ADDRESS", "ADDRESS", "LOCATION", "LOCATION", "ZIPCODE", "IP_ADDRESS"]'}


## 9. Upload Prompts and Config (Same Repo, config/ Path)

Upload prompt templates and experiment config to the same repo under config/ so benchmark and config live in EdyVision/pii-skills-ablation.

In [20]:
config_repo_id = CONFIG["config_repo"]["repo_id"]
api = HfApi()

print(f"Uploading prompts and config to: {config_repo_id}")

# Get prompt version from config (defaults to v1)
prompt_version = CONFIG["experiment"].get("prompt_version", "v1")
print(f"Prompt version: {prompt_version}")

# Upload prompts (2x2 factorial design: zero_shot, with_docs, with_tools, with_skills)
# Also upload skill_workflow.txt (SKILL part returned by view_skill in with_skills condition)
# Store in versioned paths: prompts/v1/, prompts/v2/, etc.
prompt_files = [
    "zero_shot.txt",
    "with_docs.txt",
    "with_tools.txt",
    "with_skills.txt",
    "skill_workflow.txt",
]
for filename in prompt_files:
    local_path = Path("../prompts") / filename
    if local_path.exists():
        upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=f"config/prompts/{prompt_version}/{filename}",
            repo_id=config_repo_id,
            repo_type="dataset",
            commit_message=f"Upload {filename} (v{prompt_version})",
        )
        print(f"  Uploaded: config/prompts/{prompt_version}/{filename}")
    else:
        print(f"  Not found: {local_path}")

# Upload config.yaml under config/
config_path = Path("../config.yaml")
if config_path.exists():
    upload_file(
        path_or_fileobj=str(config_path),
        path_in_repo="config/config.yaml",
        repo_id=config_repo_id,
        repo_type="dataset",
        commit_message="Upload config/config.yaml",
    )
    print(f"  Uploaded: config/config.yaml")

# Upload PII label -> PII Codex mapping under config/
mapping_path = Path("../config/pii_label_to_piicodex.json")
if mapping_path.exists():
    upload_file(
        path_or_fileobj=str(mapping_path),
        path_in_repo="config/pii_label_to_piicodex.json",
        repo_id=config_repo_id,
        repo_type="dataset",
        commit_message="Upload config/pii_label_to_piicodex.json",
    )
    print(f"  Uploaded: config/pii_label_to_piicodex.json")

print(f"\nBenchmark + config: https://huggingface.co/datasets/{config_repo_id}")

Uploading prompts and config to: EdyVision/pii-skills-ablation
Prompt version: v1
  Uploaded: config/prompts/v1/zero_shot.txt
  Uploaded: config/prompts/v1/with_docs.txt
  Uploaded: config/prompts/v1/with_tools.txt
  Uploaded: config/prompts/v1/with_skills.txt
  Uploaded: config/prompts/v1/skill_workflow.txt
  Uploaded: config/config.yaml
  Uploaded: config/pii_label_to_piicodex.json

Benchmark + config: https://huggingface.co/datasets/EdyVision/pii-skills-ablation


## 10. Summary

Data preparation complete. All resources uploaded to HuggingFace.

In [21]:
print("=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)
print(f"\nBenchmark + config: https://huggingface.co/datasets/{repo_id}")
print(
    f"Results repo:      https://huggingface.co/datasets/{CONFIG['results_repo']['repo_id']}"
)
print(f"\nNext: Run notebook 02_ablation_study.ipynb to execute the ablation study.")

DATA PREPARATION COMPLETE

Benchmark + config: https://huggingface.co/datasets/EdyVision/pii-skills-ablation
Results repo:      https://huggingface.co/datasets/EdyVision/pii-skills-ablation-results

Next: Run notebook 02_ablation_study.ipynb to execute the ablation study.
